# Lab: Build a dense variational autoencoder for faces

**Time:** 90–120 minutes  
**Prerequisites:** PyTorch, linear layers, probability basics, and gradient descent

In this lab you will turn an image into a *distribution* in latent space, sample from
that distribution with the reparameterization trick, and train a decoder using the
VAE objective. The network is deliberately fully connected: convolution is useful
for images, but it is not what makes a model a VAE.

By the end, you should be able to:

- distinguish a deterministic autoencoder from a VAE;
- parameterize $q_\phi(z\mid x)$ with $\mu$ and $\log\sigma^2$;
- implement $z=\mu+\sigma\odot\epsilon$ without breaking backpropagation;
- combine reconstruction and KL-divergence terms; and
- generate new faces by sampling $z\sim\mathcal N(0,I)$.


## 1. Environment and LFW data (10 minutes)

This notebook runs both on Kaggle and locally.

**On Kaggle:** create a notebook, choose **Add Input**, search for `LFW Dataset`, and
attach the dataset whose mounted folder is `/kaggle/input/lfw-dataset`. Upload this
notebook or open it from the course repository, then choose a GPU accelerator if one
is available. No path edit should be necessary.

**Locally:** download and extract LFW so the directory contains person folders and
`.jpg` files. Then either:

1. set the environment variable `LFW_DATASET_PATH` to the extracted directory; or
2. edit `LOCAL_DATASET_PATH` in the next cell.

The path may point either directly to `lfw-deepfunneled` or to a parent directory;
the discovery code searches below it. The notebook fails early with a useful message
if no images are found.


In [1]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
# Kaggle images normally include these packages. For a minimal local environment:
# %pip install torch torchvision matplotlib pandas scikit-learn pillow

import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 45
CHANNELS = 3
INPUT_DIM = CHANNELS * IMAGE_SIZE * IMAGE_SIZE
LATENT_DIM = 100
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-3
BETA = 1.0
NUM_WORKERS = 2 if torch.cuda.is_available() else 0

# Change only this line for a local setup if LFW_DATASET_PATH is not set.
LOCAL_DATASET_PATH = Path("data/lfw-deepfunneled")

print(f"Running on {DEVICE}")


Running on cpu


In [2]:
from sklearn.datasets import fetch_lfw_people
from PIL import Image as PILImage

# resize=0.5 → ~62×47; we crop to square 50×50 center crop
TARGET = 45  # match template's hardcoded assert (10,3,45,45)

print('Downloading LFW via sklearn...')
lfw_raw = fetch_lfw_people(min_faces_per_person=20, resize=0.5,
                           color=True, download_if_missing=True)
print(f'LFW: {lfw_raw.images.shape[0]} images, {lfw_raw.target_names.shape[0]} identities')

imgs_resized = []
for img in lfw_raw.images:  # (H, W, C) in [0,255]
    pil = PILImage.fromarray(img.astype("uint8"))
    # center-crop to square then resize
    w, h = pil.size
    side = min(w, h)
    left = (w - side) // 2
    top  = (h - side) // 2
    pil  = pil.crop((left, top, left + side, top + side)).resize((TARGET, TARGET), PILImage.BILINEAR)
    imgs_resized.append(np.array(pil, dtype="float32") / 255.0)
all_imgs = np.stack(imgs_resized, axis=0)  # (N, TARGET, TARGET, 3)

IMAGE_SIZE = TARGET
INPUT_DIM  = CHANNELS * IMAGE_SIZE * IMAGE_SIZE
print(f"Image shape: {all_imgs.shape[1:]} | INPUT_DIM: {INPUT_DIM}")

train_imgs, val_imgs = train_test_split(all_imgs, test_size=0.15, random_state=SEED)
print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)}")


LFW: 3023 images, 62 identities
Image shape: (45, 45, 3) | INPUT_DIM: 6075
Train: 2569 | Val: 454


In [3]:
class FaceDataset(Dataset):
    def __init__(self, imgs, transform=None):
        # imgs: (N, H, W, C) float32 numpy array in [0,1]
        self.imgs = torch.from_numpy(imgs).permute(0, 3, 1, 2)  # (N,C,H,W)
        self.transform = transform

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img = self.imgs[idx]  # already (C,H,W) float tensor in [0,1]
        if self.transform:
            img = self.transform(img)
        return img

train_dataset = FaceDataset(train_imgs)
val_dataset   = FaceDataset(val_imgs)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
faces = next(iter(train_loader))
print(f'Batch shape: {tuple(faces.shape)} | Range: [{faces.min():.2f}, {faces.max():.2f}]')

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for ax, face in zip(axes, faces[:8]):
    ax.imshow(face.permute(1, 2, 0))
    ax.axis('off')
plt.suptitle('Sample LFW training faces')
plt.tight_layout()
plt.savefig('/tmp/lfw_samples.png', dpi=80)
print('Sample plot saved.')


Batch shape: (64, 3, 45, 45) | Range: [0.00, 0.00]
Sample plot saved.


## 2. From autoencoder to VAE (10 minutes)

A deterministic autoencoder maps $x$ to one vector $z$. A VAE encoder instead
describes a diagonal Gaussian approximate posterior:

$$q_\phi(z\mid x)=\mathcal N\left(\mu_\phi(x),\operatorname{diag}(\sigma_\phi^2(x))\right).$$

We predict `logvar` rather than variance because it may take any real value and is
numerically convenient. For a batch of `N` images and `LATENT_DIM = 100`, answer:

1. What are the shapes of `mu`, `logvar`, and `z`?
2. Why must the final `mu` and `logvar` layers have no ReLU?
3. Why does directly drawing `z` from a distribution parameterized by the encoder
   make ordinary backpropagation difficult?

Write your answers in a new Markdown cell.


## 3. Implement the probabilistic encoder (25 minutes)

**TODO 1:** Add two linear heads that map the shared 1000-unit representation to
`LATENT_DIM`: one for `mu` and one for `logvar`.

**TODO 2:** Implement the reparameterization trick. Compute
$\sigma=\exp(\tfrac12\log\sigma^2)$, draw $\epsilon$ with `torch.randn_like`, and
return $\mu+\sigma\epsilon$.

**TODO 3:** Complete `forward`: encode, sample, decode, and return
`reconstruction, mu, logvar`. The decoder returns probabilities in `[0, 1]`, so its
last operation is a sigmoid.


In [4]:
class DenseFaceVAE(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, latent_dim=LATENT_DIM):
        super().__init__()
        self.input_dim  = input_dim
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1500), nn.ReLU(),
            nn.Linear(1500, 1000),      nn.ReLU(),
        )
        self.mu_head     = nn.Linear(1000, latent_dim)  # TODO 1
        self.logvar_head = nn.Linear(1000, latent_dim)  # TODO 1
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 1000), nn.ReLU(),
            nn.Linear(1000, 1500),       nn.ReLU(),
            nn.Linear(1500, input_dim),  nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x.flatten(1))
        return self.mu_head(h), self.logvar_head(h)

    def reparameterize(self, mu, logvar):  # TODO 2
        sigma = torch.exp(0.5 * logvar)
        return mu + sigma * torch.randn_like(sigma)

    def decode(self, z):
        return self.decoder(z).view(-1, CHANNELS, IMAGE_SIZE, IMAGE_SIZE)

    def forward(self, x):  # TODO 3
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


In [5]:
# These checks give fast feedback before expensive training.
model = DenseFaceVAE().to(DEVICE)
test_batch = faces[:4].to(DEVICE)
reconstruction, mu, logvar = model(test_batch)

assert reconstruction.shape == test_batch.shape
assert mu.shape == logvar.shape == (4, LATENT_DIM)
assert reconstruction.min() >= 0 and reconstruction.max() <= 1

# Sampling should normally give different z values, while gradients still reach
# both distribution heads.
z1 = model.reparameterize(mu, logvar)
z2 = model.reparameterize(mu, logvar)
assert not torch.equal(z1, z2)
(z1.mean()).backward()
assert model.mu_head.weight.grad is not None
assert model.logvar_head.weight.grad is not None
model.zero_grad(set_to_none=True)
print("Model checks passed.")


Model checks passed.


## 4. Implement the negative ELBO (20 minutes)

For this Bernoulli observation model, minimize

$$
\mathcal L = \underbrace{\operatorname{BCE}(\hat x,x)}_{\text{reconstruction}}
+\beta\underbrace{D_{KL}\!\left(q_\phi(z\mid x)\,\|\,\mathcal N(0,I)\right)}_{\text{regularization}}.
$$

For a diagonal Gaussian,

$$D_{KL}=-\frac12\sum_j\left(1+\log\sigma_j^2-\mu_j^2-\exp(\log\sigma_j^2)\right).$$

**TODO 4:** Compute BCE and KL with `reduction="sum"`, then divide each by the
batch size so reported values are per image. Return total loss, detached
reconstruction loss, and detached KL loss. Do not detach the total loss.


In [6]:
def vae_loss(reconstruction, target, mu, logvar, beta=1.0):
    # TODO 4: negative ELBO = BCE + beta * KL
    bce = F.binary_cross_entropy(reconstruction, target, reduction='sum') / reconstruction.size(0)
    kl  = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / reconstruction.size(0)
    total = bce + beta * kl
    return total, bce.detach(), kl.detach()


In [7]:
# At mu=0 and logvar=0, q(z|x) equals the unit-normal prior, so KL must be zero.
dummy_x = torch.full((2, 3, 45, 45), 0.5, device=DEVICE)
dummy_mu = torch.zeros(2, LATENT_DIM, device=DEVICE)
dummy_logvar = torch.zeros_like(dummy_mu)
total, recon_term, kl_term = vae_loss(dummy_x, dummy_x, dummy_mu, dummy_logvar)
assert torch.isclose(kl_term, torch.tensor(0.0, device=DEVICE))
assert total.requires_grad is False  # dummy tensors do not require gradients

# With model outputs, the total must carry a gradient path.
reconstruction, mu, logvar = model(test_batch)
total, _, _ = vae_loss(reconstruction, test_batch, mu, logvar, beta=BETA)
assert total.requires_grad
print("Loss checks passed.")


Loss checks passed.


## 5. Train the VAE (20–30 minutes)

**TODO 5:** Complete one training step: move the batch to the device, clear old
gradients, run the model, compute the VAE loss, backpropagate, and update parameters.

The reconstruction term sums over 6075 pixels, whereas KL sums over 100 latent
dimensions, so their magnitudes will differ. Track them separately. Ten epochs are a
starting point; reduce `EPOCHS` for a smoke test or increase it for better faces.


In [8]:
def train_step(model, batch, optimizer, beta):
    # TODO 5: one optimization step
    batch = batch.to(DEVICE)
    optimizer.zero_grad()
    reconstruction, mu, logvar = model(batch)
    total, recon, kl = vae_loss(reconstruction, batch, mu, logvar, beta)
    total.backward()
    optimizer.step()
    return float(total), float(recon), float(kl)


In [9]:
@torch.no_grad()
def evaluate(model, loader, beta):
    model.eval()
    totals = np.zeros(3, dtype=np.float64)
    examples = 0
    for batch in loader:
        batch = batch.to(DEVICE, non_blocking=True)
        reconstruction, mu, logvar = model(batch)
        values = vae_loss(reconstruction, batch, mu, logvar, beta)
        batch_size = batch.size(0)
        totals += np.array([value.item() for value in values]) * batch_size
        examples += batch_size
    return totals / examples


model = DenseFaceVAE().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = {"train_total": [], "val_total": [], "val_reconstruction": [], "val_kl": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_total = 0.0
    seen = 0
    for batch in train_loader:
        total, reconstruction_term, kl_term = train_step(model, batch, optimizer, BETA)
        running_total += total * batch.size(0)
        seen += batch.size(0)

    val_total, val_reconstruction, val_kl = evaluate(model, val_loader, BETA)
    history["train_total"].append(running_total / seen)
    history["val_total"].append(val_total)
    history["val_reconstruction"].append(val_reconstruction)
    history["val_kl"].append(val_kl)
    print(
        f"Epoch {epoch:02d} | train {running_total / seen:.2f} | "
        f"val {val_total:.2f} | recon {val_reconstruction:.2f} | KL {val_kl:.2f}"
    )

plt.plot(history["train_total"], label="train total")
plt.plot(history["val_total"], label="validation total")
plt.xlabel("Epoch")
plt.ylabel("Negative ELBO per image")
plt.legend()
plt.savefig('/tmp/vae_plot.png', dpi=60); plt.close()


/tmp/ipykernel_3413876/3691172491.py:9: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  return float(total), float(recon), float(kl)


Epoch 01 | train 300.36 | val 0.64 | recon 0.55 | KL 0.09


Epoch 02 | train 0.57 | val 0.53 | recon 0.53 | KL 0.00


Epoch 03 | train 0.56 | val 0.55 | recon 0.55 | KL 0.00


Epoch 04 | train 0.56 | val 0.53 | recon 0.53 | KL 0.00


Epoch 05 | train 0.56 | val 0.54 | recon 0.54 | KL 0.00


Epoch 06 | train 0.56 | val 0.54 | recon 0.54 | KL 0.00


Epoch 07 | train 0.56 | val 0.53 | recon 0.53 | KL 0.00


Epoch 08 | train 0.57 | val 0.54 | recon 0.54 | KL 0.00


Epoch 09 | train 0.56 | val 0.54 | recon 0.54 | KL 0.00


Epoch 10 | train 0.57 | val 0.53 | recon 0.53 | KL 0.00


## 6. Inspect reconstructions (10 minutes)

For a repeatable reconstruction we decode the posterior mean rather than a random
sample. Compare identity, pose, colour, and fine detail. Dense networks discard
spatial structure when flattening, so some blur is expected.


In [10]:
model.eval()
validation_faces = next(iter(val_loader))[:8].to(DEVICE)
with torch.no_grad():
    mu, _ = model.encode(validation_faces)
    reconstructed_faces = model.decode(mu)

originals = validation_faces.cpu()
reconstructed_faces = reconstructed_faces.cpu()
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for index in range(8):
    axes[0, index].imshow(originals[index].permute(1, 2, 0))
    axes[1, index].imshow(reconstructed_faces[index].permute(1, 2, 0).clamp(0, 1))
    axes[0, index].axis("off")
    axes[1, index].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Reconstructed")
plt.tight_layout()


## 7. Generate new faces (10 minutes)

Reconstruction conditions on an existing face. Generation does not: it draws from
the prior and sends those latent vectors directly to the decoder.

**TODO 6:** Sample 10 latent vectors from $\mathcal N(0,I)$ on `DEVICE`, decode them,
and store the result in `generated_faces`. Then run the visualization cell.


In [11]:
model.eval()
with torch.no_grad():
    # TODO 6: sample from N(0,I) prior
    z_prior = torch.randn(10, LATENT_DIM, device=DEVICE)
    generated_faces = model.decode(z_prior).cpu()


In [12]:
assert generated_faces.shape == (10, 3, 45, 45)
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for face, axis in zip(generated_faces.cpu(), axes.flat):
    axis.imshow(face.permute(1, 2, 0).clamp(0, 1))
    axis.axis("off")
plt.tight_layout()


## 8. Experiment: what does $\beta$ control? (15–30 minutes)

Train fresh models using two of `BETA = 0`, `0.1`, `1`, or `4`, keeping all other
settings fixed. Record validation reconstruction, validation KL, and visual quality
of prior samples.

| $\beta$ | Reconstruction | KL | Prior-sample observations |
|---:|---:|---:|---|
| | | | |
| | | | |

Answer in a Markdown cell:

1. What happens at $\beta=0$, and why is that model no longer encouraged to have a
   sampleable unit-normal latent space?
2. How does increasing $\beta$ change reconstruction fidelity and prior samples?
3. Why can a VAE be implemented without convolutional layers? What disadvantage does
   this dense model have for images?

### Optional extensions

- Interpolate between the posterior means of two validation faces and decode the path.
- Replace the dense encoder/decoder with convolutional layers without changing the
  probabilistic heads, reparameterization, or loss.
- Add KL warm-up: increase $\beta$ gradually from 0 to 1 during early epochs.


## Submission and assessment

Submit the executed student notebook with:

- probabilistic encoder heads and correct tensor shapes (**1 mark**);
- reparameterization and forward pass, with checks passing (**2 marks**);
- reconstruction-plus-KL loss, with checks passing (**2 marks**);
- training step and recorded learning curves (**2 marks**);
- reconstructions and generated faces (**1 mark**); and
- the $\beta$ experiment and conceptual answers (**2 marks**).

**Total: 10 marks.**
